# 🏠 Predicción de Precios de Viviendas con Redes Neuronales

**Caso de Estudio** — Firma de inversión inmobiliaria

**Objetivo:** Modelo RNA para predecir precios. Métricas: RMSE < 15% precio medio, R² > 0.60

**Dataset:** [Housing Prices (Kaggle)](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset)


## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo la primera vez
!pip install tensorflow numpy pandas scikit-learn matplotlib seaborn kagglehub


## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
import warnings
warnings.filterwarnings('ignore')

# Semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)
print(f'TensorFlow: {tf.__version__}')


## 2. Carga del Dataset

Descargamos el dataset de Kaggle con `kagglehub`.

In [ ]:
# Descarga automática desde Kaggle
try:
    import kagglehub, os
    path = kagglehub.dataset_download('yasserh/housing-prices-dataset')
    df = pd.read_csv(os.path.join(path, 'Housing.csv'))
    print('Dataset cargado desde kagglehub')
except Exception as e:
    print(f'kagglehub falló: {e}')
    print('Descarga Housing.csv manualmente y ponlo en esta carpeta')
    df = pd.read_csv('Housing.csv')

print(f'Dimensiones: {df.shape}')
df.head()


## 3. Exploración Inicial

Estructura, tipos, estadísticas y valores nulos.

In [ ]:
print(df.info())
print('\nEstadísticas:')
display(df.describe())
print(f'\nValores nulos: {df.isnull().sum().sum()}')


## 4. Visualización de Distribuciones

Histogramas y boxplots para detectar outliers.

In [ ]:
# Histogramas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribución de Variables Numéricas', fontsize=16, fontweight='bold')
for i, col in enumerate(['price','area','bedrooms','bathrooms','stories','parking']):
    ax = axes[i//3, i%3]
    ax.hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontweight='bold')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Media:{df[col].mean():.0f}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Boxplots - Outliers', fontsize=16, fontweight='bold')
for i, col in enumerate(['price','area','bedrooms','bathrooms']):
    sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
    axes[i].set_title(col, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. Análisis de Correlación

Matriz de correlación para identificar relaciones con `price`.

In [ ]:
# Codificación temporal para correlación
df_c = df.copy()
for c in ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']:
    df_c[c] = df_c[c].map({'yes':1,'no':0})
df_c['furnishingstatus'] = df_c['furnishingstatus'].map({'unfurnished':0,'semi-furnished':1,'furnished':2})

plt.figure(figsize=(12,10))
corr = df_c.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Matriz de Correlación', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

print('Correlación con price:')
print(corr['price'].drop('price').sort_values(ascending=False))


## 6. Preprocesamiento

- Imputar nulos (mediana/moda)
- Codificar binarias (yes/no → 1/0)
- One-hot encoding para furnishingstatus

In [ ]:
# Imputar nulos
for c in df.select_dtypes(include=[np.number]).columns:
    if df[c].isnull().sum()>0: df[c].fillna(df[c].median(), inplace=True)
for c in df.select_dtypes(include=['object']).columns:
    if df[c].isnull().sum()>0: df[c].fillna(df[c].mode()[0], inplace=True)

# Codificar binarias
for c in ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']:
    df[c] = df[c].map({'yes':1,'no':0})

# One-hot encoding
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True, dtype=int)
print(f'Columnas: {df.columns.tolist()}')
df.head()


## 7. Split Train/Test (80/20) y Normalización

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

# Normalización (fit solo en train)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Normalización aplicada')


## 8. Arquitectura de la Red Neuronal

- Entrada: n features
- 2 capas ocultas (64, 32) con ReLU + Dropout + L2
- Salida: 1 neurona lineal (regresión)

In [ ]:
n_feat = X_train_s.shape[1]

model = keras.Sequential([
    layers.Input(shape=(n_feat,)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.2),
    layers.Dense(1, activation='linear')
], name='modelo_precios')

model.summary()


## 9. Compilar Modelo

Loss=MSE, Optimizador=Adam(lr=0.001), Métrica=MAE

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
print('Modelo compilado')


## 10. Entrenamiento

100 épocas, batch=32, EarlyStopping + ReduceLR.

In [ ]:
es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
rlr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[es, rlr],
    verbose=1
)


## 11. Curvas de Entrenamiento

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,5))
ax[0].plot(history.history['loss'],label='Train',color='steelblue',lw=2)
ax[0].plot(history.history['val_loss'],label='Val',color='coral',lw=2)
ax[0].set_title('Loss (MSE)',fontweight='bold'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history.history['mae'],label='Train',color='steelblue',lw=2)
ax[1].plot(history.history['val_mae'],label='Val',color='coral',lw=2)
ax[1].set_title('MAE',fontweight='bold'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle('Curvas de Entrenamiento',fontsize=16,fontweight='bold')
plt.tight_layout(); plt.show()


## 12. Experimento: SGD con Momentum

Comparamos Adam vs SGD.

In [ ]:
model_sgd = keras.Sequential([
    layers.Input(shape=(n_feat,)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.2),
    layers.Dense(1, activation='linear')
])
model_sgd.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001,momentum=0.9), loss='mse', metrics=['mae'])
model_sgd.fit(X_train_s, y_train, validation_split=0.2, epochs=100, batch_size=32,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)], verbose=0)
print('SGD entrenado')


## 13. Evaluación en Test

In [ ]:
yp_adam = model.predict(X_test_s, verbose=0).flatten()
yp_sgd = model_sgd.predict(X_test_s, verbose=0).flatten()

rmse_a = np.sqrt(mean_squared_error(y_test, yp_adam))
r2_a = r2_score(y_test, yp_adam)
rmse_s = np.sqrt(mean_squared_error(y_test, yp_sgd))
r2_s = r2_score(y_test, yp_sgd)
umbral = 0.15 * y_test.mean()

print(f'         Adam         SGD')
print(f'RMSE  {rmse_a:>10,.0f}  {rmse_s:>10,.0f}')
print(f'R²    {r2_a:>10.4f}  {r2_s:>10.4f}')
print(f'\nUmbral RMSE (15%): {umbral:,.0f}')
print(f'Adam RMSE ok: {rmse_a < umbral}')
print(f'Adam R² > 0.60: {r2_a > 0.6}')


## 14. Gráficos de Evaluación

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(18,5))

ax[0].scatter(y_test, yp_adam, alpha=0.5, color='steelblue', s=50)
ax[0].plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--',lw=2)
ax[0].set_xlabel('Real'); ax[0].set_ylabel('Predicho')
ax[0].set_title(f'Real vs Predicho (R²={r2_a:.3f})',fontweight='bold')
ax[0].grid(alpha=0.3)

err = y_test.values - yp_adam
ax[1].hist(err, bins=30, color='steelblue', edgecolor='white')
ax[1].axvline(0, color='red', linestyle='--', lw=2)
ax[1].set_title('Distribución Errores',fontweight='bold'); ax[1].grid(alpha=0.3)

err_pct = np.abs(err/y_test.values)*100
ax[2].hist(err_pct, bins=30, color='coral', edgecolor='white')
ax[2].axvline(10, color='red', linestyle='--', lw=2, label='10%')
ax[2].set_title('Error Porcentual',fontweight='bold'); ax[2].legend(); ax[2].grid(alpha=0.3)

plt.suptitle('Evaluación',fontsize=16,fontweight='bold')
plt.tight_layout(); plt.show()


## 15. Errores Significativos (>10%)

In [ ]:
mask = err_pct > 10
print(f'Con error>10%: {mask.sum()} de {len(y_test)} ({mask.sum()/len(y_test)*100:.1f}%)')
if mask.sum()>0:
    he = pd.DataFrame({'Real':y_test.values[mask],'Pred':yp_adam[mask],'Err%':err_pct[mask].round(1)})
    display(he.sort_values('Err%',ascending=False).head(10))


## 16. Comparación con Baseline (Regresión Lineal)

In [ ]:
lr = LinearRegression().fit(X_train_s, y_train)
yp_lr = lr.predict(X_test_s)
rmse_lr = np.sqrt(mean_squared_error(y_test, yp_lr))
r2_lr = r2_score(y_test, yp_lr)

print(f'         RNA      Reg.Lin.')
print(f'RMSE  {rmse_a:>9,.0f}  {rmse_lr:>9,.0f}')
print(f'R²    {r2_a:>9.4f}  {r2_lr:>9.4f}')
if r2_a > r2_lr:
    print(f'RNA supera a Reg.Lineal en R² por {r2_a-r2_lr:.4f}')


## 17. Validación Cruzada K-Fold (k=5)

In [ ]:
X_all = scaler.fit_transform(X)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
sc = []
for fold,(ti,vi) in enumerate(kf.split(X_all)):
    m = keras.Sequential([
        layers.Input(shape=(n_feat,)),
        layers.Dense(64,activation='relu',kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(32,activation='relu',kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(1,activation='linear')])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),loss='mse')
    m.fit(X_all[ti],y.values[ti],epochs=100,batch_size=32,
        validation_data=(X_all[vi],y.values[vi]),
        callbacks=[callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)],verbose=0)
    r2=r2_score(y.values[vi],m.predict(X_all[vi],verbose=0).flatten())
    sc.append(r2)
    print(f'Fold {fold+1}: R²={r2:.4f}')
print(f'\nMedia R²: {np.mean(sc):.4f} (±{np.std(sc):.4f})')


## 18. Guardar Modelo

In [ ]:
model.save('modelo_precios_viviendas.keras')
print('Modelo guardado')
m2 = keras.models.load_model('modelo_precios_viviendas.keras')
print(f'Verificación: {m2.predict(X_test_s[:3],verbose=0).flatten()}')


## 19. Resumen Final

In [ ]:
print(f'''\n
RESUMEN FINAL\n
{'='*50}\n
Arquitectura: {n_feat} → 64(ReLU) → 32(ReLU) → 1(Linear)\n
Optimizador: Adam (lr=0.001), Loss: MSE\n
\n
Test:  RMSE={rmse_a:,.0f} (umbral:{umbral:,.0f})  R²={r2_a:.4f}\n
KFold: R² medio={np.mean(sc):.4f}\n
vs LR: RNA R²={r2_a:.4f} vs LR R²={r2_lr:.4f}\n
''')
